# SGLang 常用配置参数详解 — 新手避坑指南

**定位：** 全面解读 SGLang 服务端启动参数，帮助初学者理解每个配置项的含义、默认值和最佳实践，避免常见配置错误导致 OOM 或性能低下。

```bash
# 默认启动命令
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

> 本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台

---

## 硬件与环境要求

| 项目 | 最低要求 | 推荐配置 |
|------|---------|----------|
| GPU | NVIDIA GPU（算力 ≥ 7.0） | RTX 3060 及以上 |
| 显存 (VRAM) | ≥ 6 GB | ≥ 8 GB |
| 驱动版本 | ≥ 525.0 | 最新稳定版 |
| CUDA 版本 | ≥ 12.1 | 12.4+ |
| Python | 3.9 – 3.12 | 3.10 |
| 操作系统 | Linux / WSL2 | Ubuntu 22.04 |

---

## 依赖安装

推荐使用虚拟环境隔离依赖：

```bash
# 创建并激活虚拟环境
python -m venv sglang-env
source sglang-env/bin/activate

# 安装 SGLang 全部组件及所需依赖
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和 Python 环境正常
2. 如需安装依赖，运行「可选安装」单元格
3. 阅读参数说明表，了解每个启动参数的含义
4. 运行「基础启动」单元格启动 SGLang 服务
5. 运行「验证参数生效」单元格确认配置正确
6. 完成后运行「清理资源」单元格释放 GPU 显存

---

In [ ]:
# === 环境检查 ===
import subprocess  # 导入子进程模块
import sys  # 导入系统模块

print('=' * 50)  # 打印分隔线
print('SGLang 配置参数指南 — 环境检查')  # 打印标题
print('=' * 50)  # 打印分隔线

print(f'Python 版本: {sys.version}')  # 输出当前 Python 版本

try:  # 尝试执行 nvidia-smi 命令
    result = subprocess.run(  # 运行 nvidia-smi 查询 GPU 信息
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],  # 查询 GPU 名称和显存
        capture_output=True, text=True, timeout=10  # 捕获输出并设置超时
    )  # 命令执行完成
    if result.returncode == 0:  # 如果命令执行成功
        print(f'GPU 信息: {result.stdout.strip()}')  # 输出 GPU 信息
    else:  # 如果命令执行失败
        print('警告: nvidia-smi 执行失败')  # 输出警告
except FileNotFoundError:  # 如果找不到 nvidia-smi
    print('警告: 未找到 nvidia-smi，请确认已安装 NVIDIA 驱动')  # 提示用户

try:  # 尝试导入 sglang
    import sglang  # 导入 sglang 模块
    print(f'SGLang 版本: {sglang.__version__}')  # 输出版本号
except ImportError:  # 如果未安装
    print('SGLang 未安装，请运行下方安装单元格')  # 提示安装

try:  # 尝试导入 openai
    import openai  # 导入 openai 模块
    print(f'OpenAI SDK 版本: {openai.__version__}')  # 输出版本号
except ImportError:  # 如果未安装
    print('OpenAI SDK 未安装')  # 提示安装

print('=' * 50)  # 打印分隔线

In [ ]:
# === 可选：安装依赖（已安装可跳过） ===
# import subprocess  # 导入子进程模块
# import sys  # 导入系统模块
# subprocess.check_call([  # 执行 pip 安装命令
#     sys.executable, '-m', 'pip', 'install', '-q',  # 使用当前解释器静默安装
#     'sglang[all]', 'openai>=1.0.0', 'requests>=2.28.0'  # 安装所需依赖
# ])  # 安装命令执行完成
# print('依赖安装完成')  # 打印安装成功提示

## 第一部分：SGLang 启动参数一览

SGLang 通过 `sglang.launch_server` 模块启动推理服务，支持丰富的命令行参数来控制模型加载、内存分配、推理行为等。

**为什么配置参数很重要？**

- **显存管理：** 不合理的参数可能导致 OOM（显存溢出），合理配置可充分利用硬件资源
- **性能调优：** 正确配置 KV Cache 比例和上下文长度可以最大化吞吐量
- **兼容性：** 不同模型可能需要特定参数（如 `--trust-remote-code`）才能正常运行
- **多卡部署：** `--tp` 等参数需要与实际硬件配置匹配，设错会直接报错

## 第二部分：核心参数详解

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--model-path` | 无（必填） | 模型路径，支持 HuggingFace 模型 ID（如 `Qwen/Qwen2.5-0.5B-Instruct`）或本地目录路径 |
| `--host` | `127.0.0.1` | 服务监听地址；设为 `0.0.0.0` 可接受外部网络访问 |
| `--port` | `30000` | 服务监听端口号 |
| `--tp` | `1` | Tensor Parallel 并行度；多卡部署时设为 GPU 数量 |
| `--mem-fraction-static` | `0.88` | 分配给 KV Cache 的 GPU 显存比例（范围 0.0 ~ 1.0） |
| `--max-total-tokens` | 自动计算 | 最大总 token 数，影响并发处理能力 |
| `--context-length` | 模型默认值 | 最大上下文长度；超过模型原生上限会报错 |
| `--dtype` | `auto` | 推理数据类型：`float16`、`bfloat16`、`auto`（自动选择最优） |
| `--quantization` | 无 | 量化方法：`awq`、`gptq` 等，可大幅节省显存 |
| `--served-model-name` | 与 model-path 相同 | 自定义 API 中返回的模型名称 |
| `--trust-remote-code` | `False` | 信任 HuggingFace 上的远程自定义代码（部分国产模型必需） |
| `--disable-radix-cache` | `False` | 禁用 RadixAttention 前缀缓存（一般不建议禁用） |

## 第三部分：参数组合示例

以下通过三个典型场景演示不同的参数组合方式。

In [ ]:
# === 参数组合示例：基础启动 ===
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求模块
import sys  # 导入系统模块

MODEL_PATH = 'Qwen/Qwen2.5-0.5B-Instruct'  # 设置模型路径
HOST = '127.0.0.1'  # 设置监听地址
PORT = 30000  # 设置端口号

basic_cmd = [  # 构建基础启动命令列表
    sys.executable, '-m', 'sglang.launch_server',  # 调用 sglang 启动模块
    '--model-path', MODEL_PATH,  # 指定模型路径
    '--host', HOST,  # 指定监听地址
    '--port', str(PORT),  # 指定端口号
]  # 命令构建完成

print('基础启动命令（仅必需参数）：')  # 打印命令标题
print(' '.join(basic_cmd))  # 打印完整命令
print()  # 打印空行

server_process = subprocess.Popen(  # 以子进程方式启动 SGLang 服务
    basic_cmd,  # 传入命令列表
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE,  # 捕获标准错误输出
)  # 子进程创建完成
print(f'服务进程已启动，PID = {server_process.pid}')  # 打印进程 ID

max_wait = 300  # 最大等待时间 300 秒（含模型下载时间）
start_time = time.time()  # 记录开始时间
ready = False  # 服务就绪标志初始化为 False

while time.time() - start_time < max_wait:  # 在超时范围内持续轮询
    try:  # 尝试连接服务健康检查端点
        resp = requests.get(f'http://{HOST}:{PORT}/health', timeout=5)  # 发送健康检查请求
        if resp.status_code == 200:  # 如果返回状态码 200
            elapsed = time.time() - start_time  # 计算已用时间
            print(f'服务启动成功！耗时 {elapsed:.1f} 秒')  # 打印成功信息
            ready = True  # 标记为就绪
            break  # 退出循环
    except requests.exceptions.ConnectionError:  # 如果连接失败
        pass  # 服务尚未就绪，继续等待
    time.sleep(2)  # 每 2 秒检查一次

if not ready:  # 如果超时仍未就绪
    print(f'服务未在 {max_wait} 秒内启动，请检查日志')  # 打印超时警告

In [ ]:
# === 参数组合示例：低显存配置 ===
# 此单元格仅展示命令，不实际启动新服务

MODEL_PATH = 'Qwen/Qwen2.5-0.5B-Instruct'  # 设置模型路径

low_vram_cmd = [  # 构建低显存启动命令列表
    'python', '-m', 'sglang.launch_server',  # 调用 sglang 启动模块
    '--model-path', MODEL_PATH,  # 指定模型路径
    '--host', '127.0.0.1',  # 指定监听地址
    '--port', '30000',  # 指定端口号
    '--mem-fraction-static', '0.7',  # 降低 KV Cache 显存比例至 70%
    '--context-length', '2048',  # 限制最大上下文长度为 2048
]  # 命令构建完成

print('低显存配置命令（适合 6-8GB 显存的 GPU）：')  # 打印配置说明
print(' '.join(low_vram_cmd))  # 打印完整命令
print()  # 打印空行
print('参数说明：')  # 打印参数说明标题
print('  --mem-fraction-static 0.7  → 将 KV Cache 显存比例从默认 0.88 降至 0.7')  # 解释显存比例参数
print('  --context-length 2048      → 将上下文长度从模型默认值限制为 2048')  # 解释上下文长度参数
print()  # 打印空行
print('适用场景：')  # 打印适用场景标题
print('  ✅ 显存较小的 GPU，如 RTX 3060 (12GB) 或 RTX 4060 (8GB)')  # 说明适用 GPU
print('  ✅ 减少约 20% 的 KV Cache 显存占用')  # 说明显存节省效果
print('  ✅ 短上下文场景可降低单请求的显存开销')  # 说明上下文限制的好处
print()  # 打印空行
print('⚠️ 注意：此为展示命令，未实际启动，避免与当前运行的服务冲突')  # 提示未实际启动

In [ ]:
# === 参数组合示例：高吞吐配置 ===
# 此单元格仅展示命令，不实际启动新服务

MODEL_PATH = 'Qwen/Qwen2.5-0.5B-Instruct'  # 设置模型路径

high_throughput_cmd = [  # 构建高吞吐启动命令列表
    'python', '-m', 'sglang.launch_server',  # 调用 sglang 启动模块
    '--model-path', MODEL_PATH,  # 指定模型路径
    '--host', '127.0.0.1',  # 指定监听地址
    '--port', '30000',  # 指定端口号
    '--mem-fraction-static', '0.92',  # 提高 KV Cache 显存比例至 92%
    '--context-length', '8192',  # 设置较大的上下文长度
    '--dtype', 'float16',  # 明确指定 float16 精度
]  # 命令构建完成

print('高吞吐配置命令（适合 24GB+ 显存的 GPU）：')  # 打印配置说明
print(' '.join(high_throughput_cmd))  # 打印完整命令
print()  # 打印空行
print('参数说明：')  # 打印参数说明标题
print('  --mem-fraction-static 0.92 → 更大比例的显存分配给 KV Cache，提升并发能力')  # 解释显存比例
print('  --context-length 8192      → 支持更长的输入输出上下文')  # 解释上下文长度
print('  --dtype float16            → 使用 float16 精度以获得最佳性能')  # 解释数据类型
print()  # 打印空行
print('适用场景：')  # 打印适用场景标题
print('  ✅ 高端 GPU，如 RTX 3090 (24GB)、RTX 4090 (24GB)、A100 (40/80GB)')  # 说明适用 GPU
print('  ✅ 需要处理长文本或大批量并发请求的场景')  # 说明适用场景
print()  # 打印空行
print('⚠️ 注意：在低显存 GPU 上使用此配置可能导致 OOM 错误')  # 提示风险
print('⚠️ 注意：此为展示命令，未实际启动，避免与当前运行的服务冲突')  # 提示未实际启动

## 第四部分：新手常见避坑

以下是新手配置 SGLang 时最常犯的错误，请务必注意：

- **不要同时启动多个服务占用同一 GPU：** 多个推理服务会争抢显存，导致 OOM 或性能急剧下降。启动新服务前，务必先关闭旧服务。

- **`--tp` 数不要超过实际 GPU 数量：** 例如只有 1 张 GPU 时设置 `--tp 2` 会直接报错。`--tp` 值必须等于或小于可用 GPU 数。

- **`--context-length` 设太大会 OOM：** 上下文长度越大，KV Cache 占用的显存越多。在 6-8GB 显存的 GPU 上，建议不超过 4096。

- **`--dtype` 选错导致精度或兼容性问题：** 部分旧 GPU（如 V100 以前）不支持 `bfloat16`，建议使用 `auto` 让框架自动选择最优类型。

- **忘记 `--trust-remote-code`：** 部分国产模型（如 ChatGLM 系列）使用自定义代码，需要此参数才能正常加载。

- **端口被占用：** 如果看到 `Address already in use` 错误，说明端口已被其他进程占用，请换一个端口或先关闭占用端口的进程。

In [ ]:
# === 验证参数生效 ===
import requests  # 导入 HTTP 请求模块
import json  # 导入 JSON 模块

HOST = '127.0.0.1'  # 服务地址
PORT = 30000  # 服务端口
BASE_URL = f'http://{HOST}:{PORT}'  # 构建基础 URL

# 方法一：查询模型列表
print('=== 方法一：查询已加载模型 ===')  # 打印方法一标题
try:  # 尝试查询模型信息
    resp = requests.get(f'{BASE_URL}/v1/models', timeout=10)  # 请求模型列表接口
    models_data = resp.json()  # 解析 JSON 响应
    print(json.dumps(models_data, indent=2, ensure_ascii=False))  # 格式化打印模型信息
except Exception as e:  # 如果请求失败
    print(f'请求失败: {e}')  # 打印错误信息

print()  # 打印空行

# 方法二：查询服务器配置信息
print('=== 方法二：查询服务器配置 ===')  # 打印方法二标题
try:  # 尝试查询服务器信息
    resp = requests.get(f'{BASE_URL}/get_server_info', timeout=10)  # 请求服务器信息接口
    server_info = resp.json()  # 解析 JSON 响应
    print(json.dumps(server_info, indent=2, ensure_ascii=False))  # 格式化打印服务器信息
except Exception as e:  # 如果请求失败
    print(f'请求失败（此接口可能不存在于当前版本）: {e}')  # 打印错误提示

print()  # 打印空行

# 方法三：发送测试请求验证服务可用
print('=== 方法三：发送测试请求 ===')  # 打印方法三标题
try:  # 尝试发送测试请求
    from openai import OpenAI  # 导入 OpenAI 客户端
    client = OpenAI(base_url=f'{BASE_URL}/v1', api_key='EMPTY')  # 创建客户端实例
    response = client.chat.completions.create(  # 发送聊天补全请求
        model='Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型名称
        messages=[{"role": "user", "content": "你好，请简单介绍一下你自己。"}],  # 发送测试消息
        max_tokens=50,  # 限制输出长度
    )  # 请求完成
    print(f'模型回复: {response.choices[0].message.content}')  # 打印模型回复
    print(f'使用的模型: {response.model}')  # 打印实际使用的模型名称
except Exception as e:  # 如果请求失败
    print(f'测试请求失败: {e}')  # 打印错误信息

In [ ]:
# === 清理资源 ===
import os  # 导入操作系统模块
import signal  # 导入信号模块

try:  # 尝试终止服务进程
    server_process.terminate()  # 发送 SIGTERM 信号终止服务
    server_process.wait(timeout=10)  # 等待进程退出，最多 10 秒
    print(f'服务进程 (PID={server_process.pid}) 已终止')  # 打印终止成功信息
except NameError:  # 如果 server_process 变量未定义
    print('未找到服务进程变量，可能未启动或已在其他单元格中清理')  # 打印提示
except Exception as e:  # 其他异常
    print(f'终止进程时出错: {e}')  # 打印错误信息
    try:  # 尝试强制终止
        os.kill(server_process.pid, signal.SIGKILL)  # 发送 SIGKILL 强制终止
        print('已强制终止进程')  # 打印强制终止信息
    except Exception:  # 强制终止也失败
        print('请手动运行: kill -9 <PID> 终止进程')  # 提示手动终止

print('资源清理完成')  # 打印清理完成提示

## 常见问题排查

### 问题一：设置 `--tp 2` 但只有一张 GPU

**错误信息：** `RuntimeError: Expected at least 2 GPUs but found 1`

**解决方法：**
- 使用 `nvidia-smi` 确认可用 GPU 数量
- 将 `--tp` 设为实际 GPU 数量，单卡使用 `--tp 1`（或不设置，默认值即为 1）
- 如需多卡并行，确保所有 GPU 对当前进程可见：`CUDA_VISIBLE_DEVICES=0,1`

### 问题二：`--mem-fraction-static` 设置过高导致 OOM

**错误信息：** `torch.cuda.OutOfMemoryError: CUDA out of memory`

**解决方法：**
- 将 `--mem-fraction-static` 从默认的 0.88 降低至 0.7 ~ 0.8
- 同时降低 `--context-length`，例如设为 2048
- 确认没有其他程序占用 GPU 显存（通过 `nvidia-smi` 查看）

---

**结语：** 掌握 SGLang 的启动参数是高效使用推理服务的基础。建议从基础配置开始，根据实际硬件条件和性能需求逐步调优。遇到问题时，优先检查显存使用情况和参数兼容性。